# 조각 종류와 목표 셀의 추가 정보 분리 — Full-only
## 1. 왜 / 무엇
이전에는 목표 조건에 **조각 종류+셀 좌표**를 함께 추가했습니다. 이번에는 세 조건으로 나눕니다.

| 조건 | 조작 이력 | 조각 종류 | Human 목표 셀 |
|---|---|---|---|
| prefix_only | O | - | - |
| piece_type_prefix | O | O | - |
| oracle_goal_prefix | O | O | O |

**사용자 승인(2026-11-03)에 따라 별도 Smoke 학습을 생략합니다.** 짧은 내부 검사는 코드 보호용이며 성능 지표를 만들지 않습니다.
전체 Train/Validation에서만30epoch 학습합니다. 기존 두 조건도 재학습해 이전 Full report와 재현 대조합니다.
새 조건의 반복 결정성·동결 감사는 아닙니다. [사전 고정 명세](../../../ai-harness/ml-experiments/experiments/human-goal-information-ablation.md)를 먼저 읽으세요.


## 2. 예제와 해석 범위
관측 prefix가 `move_left → rotate`로 같을 때:
- 조각 종류 조건은 **T 조각이다**만 알려줍니다.
- 목표 셀 조건은 여기에 **사람이 선택한 네 셀**까지 알려줍니다.

신경망·학습량·표본·시간 support는 동일합니다. 목표 셀 추가 전후 차이를 봅니다.
정답 목표를 알려준 진단이지, AI가 스스로 목적지를 고르는 모델/실제 플레이가 아닙니다.
보드 snapshot·미래 행동·승패는 입력에 없고, Hold·첫 행동 전 대기시간도 미지원입니다.


In [ ]:
import copy
import hashlib
import json
import os
import platform
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
EXPERIMENT_ID = "human-goal-information-ablation"
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_goal_information_ablation.ipynb"
BASE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_goal_conditioned_prefix.ipynb"
BASE_DEFS_SHA = "12f07e2a25b35fdaedd9c3c6ed805e9b6fbb6f5b3fd236569d0363b654b051d2"
REFERENCE_SHA = "d4a5ef0101fb0c6796894c4143a98af3f67aa91eaf9e15ba5180dc1c6ac6284d"
REFERENCE_PATH = ROOT / "data/ml-experiments/human-goal-conditioned-prefix/full-report-run1.json"
definition_cells = [{"id": c["id"], "source": "".join(c["source"])}
                    for c in json.loads(BASE_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
                    if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]
definition_bytes = (json.dumps(definition_cells, ensure_ascii=False, sort_keys=True,
                              allow_nan=False, indent=2) + "\n").encode("utf-8")
assert hashlib.sha256(definition_bytes).hexdigest() == BASE_DEFS_SHA
legacy = {"__name__": "goal_ablation_shared"}
REUSE_IDS = {"shared-definitions", "feature-definitions", "input-definitions",
             "model-definitions", "metric-definitions"}
for cell in definition_cells:
    if cell["id"] in REUSE_IDS:
        exec(compile(cell["source"], str(BASE_NOTEBOOK), "exec"), legacy)
for name in ("require", "json_bytes", "sha256_file", "definition_sha", "checked_paths",
             "publish_report", "load_views", "arrays", "train_variants", "fit_head",
             "evaluate_model", "threadpool_limits"):
    globals()[name] = legacy[name]
legacy["dependency"]["link"]["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
VARIANTS = ("prefix_only", "piece_type_prefix", "oracle_goal_prefix")
COMPARISONS = {"piece_minus_prefix": ("prefix_only", "piece_type_prefix"),
               "goal_minus_piece": ("piece_type_prefix", "oracle_goal_prefix"),
               "goal_minus_prefix": ("prefix_only", "oracle_goal_prefix")}


In [ ]:
def piece_type_features(standardized):
    """prefix86 + pieceType7은 유지하고 표준화된 좌표8만 제거한다."""
    result = standardized.copy()
    result[:, -8:] = 0
    return result

def piece_type_rows(rows, scaler):
    """기존 evaluator를 그대로 쓰기 위해 raw 좌표를 Train mean으로 projection한다."""
    result = []
    for row in rows:
        projected = row["x"].copy()
        projected[-8:] = scaler.mean_[-8:]
        result.append({**row, "x": projected})
    return result

def compare_results(left, right):
    delta = {k: right["gameMacro"][k] - left["gameMacro"][k] for k in left["gameMacro"]}
    games = []
    for a, b in zip(left["perGame"], right["perGame"]):
        require(a["gameId"] == b["gameId"], "GAME_EVALUATION_ORDER_CHANGED")
        games.append({"gameId": a["gameId"], **{
            k: b[k] - a[k] for k in delta if a[k] is not None and b[k] is not None}})
    return {"gameMacroDelta": delta, "perGameDelta": games}

def validate_reference(reference, provenance):
    require(reference["experimentId"] == "human-goal-conditioned-prefix"
            and reference["runMode"] == "full" and reference["fullCohortCovered"] is True
            and reference["sourceDefinitionSha256"] == BASE_DEFS_SHA
            and reference["sourceUnchanged"] is True
            and reference["training"]["epochs"] == 30
            and reference["training"]["seeds"] == [1729, 1730]
            and reference["provenance"] == provenance, "REFERENCE_CONTRACT_MISMATCH")

def check_reproduction(results, models, reference):
    errors = {}
    for variant in ("prefix_only", "oracle_goal_prefix"):
        actual, expected = results[variant], reference["results"][variant]
        require(actual["actionExamples"] == expected["actionExamples"]
                and actual["timingExamples"] == expected["timingExamples"]
                and actual["confusionMatrix"] == expected["confusionMatrix"], "REFERENCE_COUNTS_MISMATCH")
        difference = max(abs(actual["gameMacro"][k] - expected["gameMacro"][k]) for k in actual["gameMacro"])
        require(difference <= 1e-12, "REFERENCE_METRIC_NOT_REPRODUCED")
        for head in ("action", "timing"):
            require(np.allclose(models[variant]["trainLoss"][head], reference["trainingLoss"][variant][head],
                                atol=1e-12, rtol=0), "REFERENCE_LOSS_NOT_REPRODUCED")
        errors[variant] = difference
    return errors


In [ ]:
def run_full():
    require((platform.python_version(), np.__version__, legacy["sklearn"].__version__)
            == ("3.12.13", "2.5.1", "1.9.0"), "PINNED_TRAINING_ENVIRONMENT_REQUIRED")
    require(sha256_file(REFERENCE_PATH) == REFERENCE_SHA, "REFERENCE_SHA_MISMATCH")
    reference = json.loads(REFERENCE_PATH.read_text(encoding="utf-8"))
    before_code = definition_sha(SOURCE_NOTEBOOK)
    before_base = sha256_file(BASE_NOTEBOOK)
    data, provenance, watched = load_views()
    validate_reference(reference, provenance)
    print("Full cohort validated: Train149 / Validation31 games; fitting 6 models ×30 epochs.")
    models, scaler, support, training = train_variants(data["train"], 30)
    x, y, times = arrays(data["train"])
    features = piece_type_features(scaler.transform(x))
    mask = np.isfinite(times)
    time_y = np.argmin(abs(times[mask, None] - support), axis=1)
    with threadpool_limits(limits=1):
        action, action_loss = fit_head(features, y, np.arange(6), 1729, 30)
        timing, timing_loss = fit_head(features[mask], time_y, np.arange(len(support)), 1730, 30)
    models["piece_type_prefix"] = {"action": action, "timing": timing,
                                 "trainLoss": {"action": action_loss, "timing": timing_loss}}
    evaluation_rows = piece_type_rows(data["validation"], scaler)
    check_x = arrays(evaluation_rows)[0]
    require(np.allclose(scaler.transform(check_x), piece_type_features(
        scaler.transform(arrays(data["validation"])[0])), atol=1e-12, rtol=0), "MASK_PROJECTION_MISMATCH")
    results = {v: evaluate_model(models[v], scaler, support,
                                evaluation_rows if v == "piece_type_prefix" else data["validation"], v)
               for v in VARIANTS}
    for head in ("action", "timing"):
        shapes = [[w.shape for w in models[v][head].coefs_] for v in VARIANTS]
        require(all(shape == shapes[0] for shape in shapes), "ARCHITECTURE_MISMATCH")
    reproduction = check_reproduction(results, models, reference)
    comparisons = {k: compare_results(results[a], results[b]) for k, (a, b) in COMPARISONS.items()}
    delta = comparisons["goal_minus_piece"]["gameMacroDelta"]
    status = ("goal_cells_additional_signal_observed" if delta["actionNll"] < -1e-12
              and delta["macroF1"] >= -1e-12 and delta["crpsMs"] <= 1e-9 else
              "goal_cells_tradeoff_requires_review" if delta["actionNll"] < -1e-12 else "no_clear_goal_cells_gain")
    require(all(sha256_file(p) == digest for p, digest in watched.items())
            and sha256_file(REFERENCE_PATH) == REFERENCE_SHA
            and sha256_file(BASE_NOTEBOOK) == before_base
            and definition_sha(SOURCE_NOTEBOOK) == before_code
            and definition_sha(BASE_NOTEBOOK) == BASE_DEFS_SHA, "INPUT_OR_CODE_CHANGED")
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID,
            "runMode": "full", "executionPolicy": "full-only", "fullCohortCovered": True,
            "policyDecisionEligible": True, "canonicalResultEligible": False,
            "productionIntegration": False, "wholePolicyEvidence": False,
            "humanPrefixProvided": True, "holdSupported": False, "firstWaitSupported": False,
            "boardSnapshotUsed": False, "modelFitCount": 6, "smokeFitCount": 0,
            "analysisStatus": status, "decisionScope": "offline_information_ablation_only",
            "results": results, "comparisons": comparisons, "training": training,
            "trainingLoss": {v: models[v]["trainLoss"] for v in VARIANTS},
            "referenceReportSha256": REFERENCE_SHA, "referenceReproduced": True,
            "referenceMacroMaxAbsoluteError": reproduction,
            "sourceDefinitionSha256": before_code, "baseDefinitionSha256": BASE_DEFS_SHA,
            "sourceUnchanged": True, "provenance": provenance,
            "environment": reference["environment"]}

def report_path(name):
    require(isinstance(name, str) and bool(name), "NEW_FULL_REPORT_PATH_REQUIRED")
    result = Path(name)
    if not result.is_absolute():
        result = ROOT / result
    allowed = ROOT / "data/ml-experiments" / EXPERIMENT_ID
    require(".." not in result.parts and not any(p.is_symlink() for p in (result, *result.parents)),
            "OUTPUT_ALIAS_FORBIDDEN")
    require(result.parent == allowed or result.parent == Path("/private/tmp")
            or str(result.parent).startswith("/private/tmp/"), "OUTPUT_LOCATION_NOT_ALLOWED")
    require(not result.exists() and not result.is_symlink() and result.suffix == ".json",
            "REPORT_ALREADY_EXISTS_OR_INVALID")
    result.parent.mkdir(parents=True, exist_ok=True)
    return checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(result), "full")[1]


In [ ]:
def internal_checks():
    """작은 배열·경로의 correctness만 확인하며 모델 fit·성능 평가는 하지 않는다."""
    x = np.arange(303, dtype=float).reshape(3, 101)
    original = x.copy()
    masked = piece_type_features(x)
    assert np.array_equal(masked[:, :93], x[:, :93]) and not masked[:, -8:].any()
    assert np.array_equal(x, original)
    scaler = legacy["StandardScaler"]().fit(x)
    rows = [{"x": row, "action": 1, "time": 0, "gameId": "check"} for row in x]
    projected = piece_type_rows(rows, scaler)
    assert np.allclose(scaler.transform(np.stack([r["x"] for r in projected])),
                       piece_type_features(scaler.transform(x)), atol=1e-12, rtol=0)
    assert np.array_equal(x, original) and all(r["time"] == 0 for r in projected)
    p, support = np.array([[0.25, 0.75]]), np.array([0.0, 150.0])
    brute = sum(p[0] * abs(support - 75)) - 0.5 * sum(
        p[0, i] * p[0, j] * abs(a - b) for i, a in enumerate(support) for j, b in enumerate(support))
    assert np.allclose(legacy["discrete_crps"](p, support, np.array([75.])), [brute])
    with tempfile.TemporaryDirectory(dir="/private/tmp") as directory:
        path = report_path(str(Path(directory) / "check.json"))
        publish_report({"internalCheckOnly": True}, path)
        for alias in ("alias.json", "dangling.json", "hardlink.json"):
            dest = path.parent / alias
            if alias == "hardlink.json":
                os.link(path, dest)
            else:
                dest.symlink_to(path if alias == "alias.json" else path.parent / "missing.json")
        for candidate in (path, *(path.parent / name for name in ("alias.json", "dangling.json", "hardlink.json"))):
            try:
                report_path(str(candidate))
            except ValueError:
                pass
            else:
                raise AssertionError("OUTPUT_OVERWRITE_OR_ALIAS_ALLOWED")
    return "PASS: feature mask / same prefix+piece type / Train-mean projection / no mutation / CRPS / exclusive output"


In [ ]:
def show_results(report):
    print(report["analysisStatus"], "| Full-only / oracle diagnosis / NOT autonomous gameplay")
    display(pd.DataFrame({v: report["results"][v]["gameMacro"] for v in VARIANTS}).T)
    display(pd.DataFrame({k: value["gameMacroDelta"] for k, value in report["comparisons"].items()}).T)
    display(pd.DataFrame([{**row, "variant": v}
                         for v in VARIANTS for row in report["results"][v]["perAction"]]))
    display(pd.DataFrame(report["comparisons"]["goal_minus_piece"]["perGameDelta"]))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for v in VARIANTS:
        axes[0].plot(report["trainingLoss"][v]["action"], label=v)
    axes[0].set(xlabel="epoch", ylabel="Train action CE + L2", title="Fixed 30-epoch optimization")
    axes[0].legend(fontsize=7)
    actions = pd.DataFrame([{**a, "variant": v} for v in VARIANTS for a in report["results"][v]["perAction"]])
    actions.pivot(index="action", columns="variant", values="recall").plot.bar(ax=axes[1], rot=30)
    axes[1].set(ylabel="Validation recall", ylim=(0, 1), title="Per-action prediction")
    for i, key in enumerate(("piece_minus_prefix", "goal_minus_piece")):
        values = [g["actionNll"] for g in report["comparisons"][key]["perGameDelta"]]
        axes[2].bar(np.arange(len(values)) + i * 0.4, values, width=0.4, label=key)
    axes[2].axhline(0, color="black", linewidth=1)
    axes[2].set(xlabel="Validation game", ylabel="NLL difference", title="Below 0 favors added information")
    axes[2].legend(fontsize=7)
    plt.tight_layout(); plt.show(); plt.close(fig)
    print("Existing conditions reproduced:", report["referenceMacroMaxAbsoluteError"])
    print("Time support:", report["training"]["timeSupportMs"])


## 3. Full 설정·실행
아래 기본값은 비활성입니다. 이번 승인된 Full은 환경변수 또는 이 설정으로 실행합니다.

```python
RUN_FULL = False
FULL_REPORT = "data/ml-experiments/human-goal-information-ablation/full-report-run1.json"
```

이미 존재하는 report를 다시 사용하지 않습니다. 결과를 다시 보려면 4절 로딩 셀만 사용하세요.
Smoke 학습이나 작은 표본 성능 표는 없습니다. 모델은 최종30epoch에서 한 번 평가합니다.


In [ ]:
RUN_FULL = os.environ.get("HUMAN_GOAL_ABLATION_RUN_FULL", "0") == "1"
FULL_REPORT = os.environ.get("HUMAN_GOAL_ABLATION_FULL_REPORT") or None


In [ ]:
print(internal_checks())
full_report = None
if RUN_FULL:
    output = report_path(FULL_REPORT)
    full_report = run_full()
    publish_report(full_report, output)
    print("Full report saved:", str(output), "| SHA-256:", sha256_file(output))
else:
    print("Full is disabled. Only internal checks ran; no model fit or performance evaluation.")


## 4. 결과 / 기존 report 다시 읽기
학습 없이 결과만 다시 보려면 아래 REPORT_TO_READ에 저장된 JSON 경로를 적고 로딩·결과 셀만 실행합니다.



In [ ]:
REPORT_TO_READ = None
if REPORT_TO_READ:
    existing = Path(REPORT_TO_READ)
    if not existing.is_absolute():
        existing = ROOT / existing
    loaded = json.loads(existing.read_text(encoding="utf-8"))
    require(loaded["experimentId"] == EXPERIMENT_ID and loaded["runMode"] == "full"
            and loaded["baseDefinitionSha256"] == BASE_DEFS_SHA
            and loaded["referenceReportSha256"] == REFERENCE_SHA
            and loaded["sourceDefinitionSha256"] == definition_sha(SOURCE_NOTEBOOK), "STALE_REPORT")
    full_report = loaded


In [ ]:
if full_report is not None:
    show_results(full_report)


## 5. 해석·한계·다음 gate
- 첫 번째 차이는 조각 종류 추가, 두 번째 차이는 목표 셀 추가 효과입니다. 게임별 편차와 하드드롭/회전의 precision·recall을 함께 봅니다.
- NLL/CRPS는 낮을수록, F1/accuracy는 높을수록 좋습니다. 효과를 인과적 기여율이나 특정 개인 모방의 증거로 부르지 않습니다.
- 정답 목표와 Human prefix를 제공한 one-step 진단입니다. 목표 좌표 효과가 작아도 목표가 불필요하다는 뜻은 아닙니다.
- 학습 loss는 최적화 진단이며 수렴을 보장하지 않습니다. 시간은 같은16개 support의 제약을 받습니다.
- 이전 두 조건은 기존 Full과 비교하지만 새 조건은1회뿐입니다. 통계적 유의성/반복 결정성/동결 감사는 미검증입니다.
- Human 배치 모델·예측 목표를 쓴 자체 실행·Hold·첫 대기시간·현재 ruleset 호환성은 다음 별도 과제입니다. 새 수집·runtime·다음 실험을 자동 시작하지 않습니다.
